# Progressive Missingness Analysis

Notebook for multimodal progressive missingness analysis on the current `training_runs` layout.
The replicate definition depends on `RETRAIN_OUTER` and `USE_ENSEMBLE`:

- `RETRAIN_OUTER = False`: each retained `(seed, outer_fold, inner_model_k)` predictor is treated as one trained-model replicate.
- `RETRAIN_OUTER = True`: each `(seed, outer_fold)` outer-refit model is treated as one trained-model replicate.
- `USE_ENSEMBLE = True`: the probability-averaged ensemble stored in `ensemble_prob` is treated as one `(seed, outer_fold)` replicate.

Configuration in the first code cell:
- `DATASET_NAME`
- `LABEL_NAME`
- `TRAIN_DEGRADING_MODALITY`
- `RETRAIN_OUTER`
- `USE_ENSEMBLE`
- `DISTILLATION_MODEL_NAMES`: optional extra legacy/custom names to treat as distillation methods. Methods ending in `_KD` are detected automatically and excluded from **Train-time AUPMC** and **Train degradation coefficient** because train-time missingness is applied to the student while the teacher uses complete modality information

Expected results path:
- `results/<DATASET_NAME>_<LABEL_NAME>/training_runs`
- the notebook filters runs by the `_retraintrue` / `_retrainfalse` tag in the run directory name

Implemented sections:

1. **Results visualization**
- computes replicate-level AUCs using the replicate definition implied by `RETRAIN_OUTER` and `USE_ENSEMBLE`
- builds mean AUC heatmaps across `(train_missing_prop, test_missing_prop)` cells
- runs a global Friedman test across models
- marks the full heatmap figure when the global Friedman test is significant

2. **Method-level metrics**
- plots the three method-level curves before displaying the tables: train-time missingness, test-time missingness, and the best fixed train-missingness curve
- builds a method-level table with technical AUPMC metrics and degradation coefficients:
  - **Baseline AUC**: AUC at `train_missing = 0` and `test_missing = 0`
  - **Train-time AUPMC**: AUPMC over `train_missing_prop` with `test_missing_prop = 0`; distillation methods are not included
  - **Test-time AUPMC**: AUPMC over `test_missing_prop` with `train_missing_prop = 0`
  - **Best fixed-train AUPMC**: maximum AUPMC across train-missingness settings for a model, computed by fixing one `train_missing_prop` and integrating its AUC curve over `test_missing_prop`
- reports three degradation coefficients computed as normalized positive degradation area. A method can occasionally perform better than its complete-data baseline at some missingness level; those points contribute 0 because only the area where baseline/performance is above 1 is integrated:
  - **Train degradation coefficient**: distillation methods are not included
  - **Test degradation coefficient**
  - **Minimum degradation coefficient**
- for distillation methods, train-time missingness in `both` conditions and in the best fixed train-missingness curve refers to student missingness; the teacher branch receives complete modality information
- adds bootstrap 95% CIs for AUPMC and degradation coefficients in `method_level_metrics.csv`; rankings are still based only on the mean metric values
- shows a second table where each metric column lists methods ordered from best to worst for that metric
- includes line plots for both raw mean AUC curves and pointwise degradation curves; in both cases the Y axis is data-adaptive

3. **Condition-level metrics**
- computes pairwise Wilcoxon signed-rank tests with Benjamini-Hochberg FDR correction inside each `(train_missing_prop, test_missing_prop)` condition
- plots pairwise significant `ΔAUC` matrices by condition, ordering methods within each panel by condition-specific mean AUC
- summarizes the top equivalent group against the first significantly lower-ranked method within each condition
- writes a final general summary with the best complete-data, train-time, test-time, best fixed-train, and most flexible methods


In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'analysis' else cwd

# DATASET_NAME = 'mmImmuno'
# LABEL_NAME = 'OS_9_label'
DATASET_NAME = 'mmColorectal'
LABEL_NAME = 'OS_21_label'
# DATASET_NAME = 'mmProstate'
# LABEL_NAME = 'OS_27_label'

DISPLAY_DATASET_NAME = DATASET_NAME

TRAIN_DEGRADING_MODALITY = 'GLOBAL'
RETRAIN_OUTER = False
USE_ENSEMBLE = False
DISTILLATION_MODEL_NAMES = []  # _KD methods are detected automatically

N_BOOTSTRAP = 2000
BOOTSTRAP_CONFIDENCE = 0.95
BOOTSTRAP_RANDOM_SEED = 42

RESULTS_TAG = f'{DATASET_NAME}_{LABEL_NAME}' if str(LABEL_NAME).strip() else DATASET_NAME
RESULTS_ROOT = PROJECT_ROOT / 'results' / RESULTS_TAG / 'training_runs'
if not RESULTS_ROOT.exists():
    raise FileNotFoundError(
        f'Results root does not exist for dataset={DATASET_NAME!r}, label={LABEL_NAME!r}: {RESULTS_ROOT}'
    )
if not RESULTS_ROOT.is_dir():
    raise NotADirectoryError(f'Expected a directory at: {RESULTS_ROOT}')

RETRAIN_TAG = f"retrain{str(bool(RETRAIN_OUTER)).lower()}"
PREDICTION_TAG = 'ensemble' if bool(USE_ENSEMBLE) else 'inner_models'
OUTPUT_DIR = (
    (cwd / 'progressive_missingness_analysis_outputs') if cwd.name == 'analysis' else (cwd / 'analysis' / 'progressive_missingness_analysis_outputs')
) / RESULTS_TAG / TRAIN_DEGRADING_MODALITY.lower() / RETRAIN_TAG / PREDICTION_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = OUTPUT_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINT_NAME = LABEL_NAME if str(LABEL_NAME).strip() else RESULTS_ROOT.parent.name.replace(f'{DATASET_NAME}_', '', 1)

RESULTS_ROOT


In [ ]:
import sys
import importlib
import pandas as pd

ANALYSIS_DIR = cwd if cwd.name == 'analysis' else (cwd / 'analysis')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import results_analysis as ra
ra = importlib.reload(ra)

HAVE_MPL = ra.HAVE_MPL
resolve_requested_model_names = ra.resolve_requested_model_names
load_all_test_predictions = ra.load_all_test_predictions
expand_inner_model_predictions = ra.expand_inner_model_predictions
aggregate_member_patient_predictions = ra.aggregate_member_patient_predictions
build_replicate_auc_table = ra.build_replicate_auc_table
build_level1_summary = ra.build_level1_summary
build_method_plot_summary = ra.build_method_plot_summary
build_method_level_metrics = ra.build_method_level_metrics
build_degradation_curve_summary = ra.build_degradation_curve_summary
build_metric_ordering_table = ra.build_metric_ordering_table
plot_method_line_triplet = ra.plot_method_line_triplet
compute_level1_global_friedman = ra.compute_level1_global_friedman
plot_level1_auc_heatmaps = ra.plot_level1_auc_heatmaps
compute_level2_pairwise_tests = ra.compute_level2_pairwise_tests
select_level2_plot_pairs = ra.select_level2_plot_pairs
build_top_equivalent_group_counts = ra.build_top_equivalent_group_counts
build_general_results_summary = ra.build_general_results_summary
plot_level2_significant_pairs_heatmap = ra.plot_level2_significant_pairs_heatmap
plot_level3_pairwise_condition_matrices = ra.plot_level3_pairwise_condition_matrices


## Results visualization

Mean AUC heatmaps across train/test missingness conditions, with a global Friedman test across models.

Each heatmap cell reports `mean AUC ± 95% CI`. A `*` at the figure level indicates that the global Friedman test is significant (`p < 0.05`).


In [ ]:
REQUESTED_MODEL_NAMES = resolve_requested_model_names(
    results_root=RESULTS_ROOT,
    dataset_name=DATASET_NAME,
    train_degrading_modality=TRAIN_DEGRADING_MODALITY,
    retrain_outer=RETRAIN_OUTER,
)
if not REQUESTED_MODEL_NAMES:
    raise ValueError('No model folders were detected for the requested dataset / location / retrain flag.')

raw_predictions_df, missing_prediction_files = load_all_test_predictions(
    results_root=RESULTS_ROOT,
    dataset_name=DATASET_NAME,
    train_degrading_modality=TRAIN_DEGRADING_MODALITY,
    model_names=REQUESTED_MODEL_NAMES,
    retrain_outer=RETRAIN_OUTER,
    use_ensemble=USE_ENSEMBLE,
)
if raw_predictions_df.empty:
    raise ValueError('No test_predictions.csv files with usable prediction probabilities were found.')

member_prediction_df = expand_inner_model_predictions(raw_predictions_df, use_ensemble=USE_ENSEMBLE)
member_patient_df = aggregate_member_patient_predictions(member_prediction_df)
replicate_auc_df = build_replicate_auc_table(member_patient_df)
level1_summary_df = build_level1_summary(replicate_auc_df=replicate_auc_df)
level1_global_friedman_df = compute_level1_global_friedman(level1_summary_df)
method_plot_summary_df = build_method_plot_summary(
    replicate_auc_df=replicate_auc_df,
    n_bootstrap=N_BOOTSTRAP,
    confidence=BOOTSTRAP_CONFIDENCE,
    random_seed=BOOTSTRAP_RANDOM_SEED,
)
degradation_curve_summary_df = build_degradation_curve_summary(
    method_plot_summary_df=method_plot_summary_df,
    distillation_model_names=DISTILLATION_MODEL_NAMES,
)
method_level_metrics_df, best_fixed_train_curve_df = build_method_level_metrics(
    replicate_auc_df=replicate_auc_df,
    level1_df=level1_summary_df,
    distillation_model_names=DISTILLATION_MODEL_NAMES,
    n_bootstrap=N_BOOTSTRAP,
    confidence=BOOTSTRAP_CONFIDENCE,
    random_seed=BOOTSTRAP_RANDOM_SEED,
)
method_metric_ordering_df = build_metric_ordering_table(
    method_level_metrics_df,
    distillation_model_names=DISTILLATION_MODEL_NAMES,
)

replicate_auc_path = OUTPUT_DIR / 'replicate_auc_table.csv'
method_condition_summary_path = OUTPUT_DIR / 'method_condition_mean_auc_summary.csv'
level1_global_friedman_path = OUTPUT_DIR / 'level1_global_friedman.csv'
method_plot_summary_path = OUTPUT_DIR / 'method_plot_summary.csv'
degradation_curve_summary_path = OUTPUT_DIR / 'degradation_curve_summary.csv'
method_level_metrics_path = OUTPUT_DIR / 'method_level_metrics.csv'
method_metric_ordering_path = OUTPUT_DIR / 'method_metric_orderings.csv'
best_fixed_train_curve_path = OUTPUT_DIR / 'best_fixed_train_curve.csv'

replicate_auc_df.to_csv(replicate_auc_path, index=False)
level1_summary_df.to_csv(method_condition_summary_path, index=False)
level1_global_friedman_df.to_csv(level1_global_friedman_path, index=False)
method_plot_summary_df.to_csv(method_plot_summary_path, index=False)
degradation_curve_summary_df.to_csv(degradation_curve_summary_path, index=False)
method_level_metrics_df.to_csv(method_level_metrics_path, index=False)
method_metric_ordering_df.to_csv(method_metric_ordering_path, index=False)
best_fixed_train_curve_df.to_csv(best_fixed_train_curve_path, index=False)

replicate_unit_label = 'seed x outer_fold ensemble' if bool(USE_ENSEMBLE) else ('seed x outer_fold' if bool(RETRAIN_OUTER) else 'seed x outer_fold x inner_model_k')
print(f'Replicate unit: {replicate_unit_label}')
print('Saved:', replicate_auc_path)
print('Saved:', method_condition_summary_path)
print('Saved:', level1_global_friedman_path)
print('Saved:', method_plot_summary_path)
print('Saved:', degradation_curve_summary_path)
print('Saved:', method_level_metrics_path)
print('Saved:', method_metric_ordering_path)
print('Saved:', best_fixed_train_curve_path)


In [ ]:
print('Global Friedman test across heatmap cells:')
display(level1_global_friedman_df)

level1_model_count_tag = f"{level1_summary_df['model_name'].nunique()}models" if not level1_summary_df.empty else '0models'
if HAVE_MPL:
    plot_level1_auc_heatmaps(
        summary_df=method_plot_summary_df,
        friedman_global_df=level1_global_friedman_df,
        title=f'AUC ± 95% CI Heatmaps | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME} | degrading_modality = {TRAIN_DEGRADING_MODALITY}',
        figures_dir=FIGURES_DIR,
        file_name=f'level1_{level1_model_count_tag}.png',
    )
else:
    print('Matplotlib not available; skipping level 1 heatmap plot.')


## Method-level metrics

The method-level section first plots the curves and then reports the numerical metrics derived from those curves. This makes the relationship between the visual trajectory and the final scalar summary explicit.

`Baseline AUC` is the AUC obtained when both training and test data are complete (`train_missing_prop = 0`, `test_missing_prop = 0`).

`Train-time AUPMC` is the normalized area under the performance-missingness curve across train-time missingness while test data remain complete. It summarizes the curve `AUC(m_train, 0)`. Methods ending in `_KD`, plus any optional names listed in `DISTILLATION_MODEL_NAMES`, are excluded from this metric because the teacher uses complete modality information while missingness is applied to the student.

`Test-time AUPMC` is the normalized area under the curve `AUC(0, m_test)`. It measures performance when the model is trained on complete data and evaluated under increasing test-time missingness.

`Best fixed-train AUPMC` is the maximum normalized area under the test-time missingness curve obtained by fixing one train-time missingness setting for a model. It summarizes the best achievable robustness after selecting a single training missingness regime, rather than changing the regime separately for each test missingness level.

For each trajectory, the degradation curve is first computed as `baseline / AUC(missingness condition)`. Train-time and test-time curves use the complete-data baseline `(train=0, test=0)`, while the best fixed-train curve uses the selected fixed-train baseline `(train=m_train*, test=0)`. The curve is equal to 1 when performance matches the relevant baseline, above 1 when performance is worse than baseline, and below 1 when performance is better than baseline. Since the degradation coefficient is intended to measure performance loss only, the scalar coefficient integrates only the positive area above 1, equivalent to `max(baseline / AUC - 1, 0)`. In other words, missingness levels where the method improves over the relevant baseline contribute 0 and do not compensate degradation at other levels.

The degradation coefficient is then computed as the normalized positive degradation area. A value of 0 indicates no positive degradation relative to the complete-data baseline, whereas larger values indicate increasing relative performance loss. `Train degradation coefficient` is also excluded for distillation methods for the same reason as `Train-time AUPMC`.

The line plots show both the raw mean AUC curves and the raw pointwise degradation-ratio curves; scalar degradation coefficients are computed only from the area above 1.

Bootstrap 95% CIs are also computed for AUPMC and degradation coefficients by re-sampling replicate identifiers and recomputing each integrated metric. Rankings are based only on the mean metric values, not on CI bounds.

For `<method>_KD` variants, train-time missingness in `both` conditions and in the best fixed train-missingness curve should be interpreted as student missingness. The teacher branch receives complete modality information.


In [ ]:
if HAVE_MPL:
    plot_method_line_triplet(
        summary_df=method_plot_summary_df,
        metric_col='mean_auc',
        title=f'Mean AUC Curves | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME} | degrading_modality = {TRAIN_DEGRADING_MODALITY}',
        ylabel='Mean AUC',
        figures_dir=FIGURES_DIR,
        file_name=f'method_level_mean_auc_curves_{RESULTS_TAG}_{TRAIN_DEGRADING_MODALITY.lower()}.png',
        panel_titles={
            'train': 'Train-time missingness',
            'test': 'Test-time missingness',
            'envelope': 'Best fixed train-missingness',
        },
        y_limits=None,
        clip_ci=False,
    )
    plot_method_line_triplet(
        summary_df=degradation_curve_summary_df,
        metric_col='degradation_ratio',
        title=f'Degradation Curves | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME} | degrading_modality = {TRAIN_DEGRADING_MODALITY}',
        ylabel='baseline AUC / AUC',
        figures_dir=FIGURES_DIR,
        file_name=f'method_level_degradation_curves_{RESULTS_TAG}_{TRAIN_DEGRADING_MODALITY.lower()}.png',
        ci_col='degradation_ratio_ci95',
        panel_titles={
            'train': 'Train-time missingness',
            'test': 'Test-time missingness',
            'envelope': 'Best fixed train-missingness',
        },
        y_limits=None,
        clip_ci=False,
    )
else:
    print('Matplotlib not available; skipping method-level line plots.')

method_level_display_df = (
    method_level_metrics_df[[
        'model_name',
        'is_distillation_method',
        'baseline_auc',
        'train_time_aupmc',
        'train_time_aupmc_ci95_lower',
        'train_time_aupmc_ci95_upper',
        'train_degradation_coefficient',
        'train_degradation_coefficient_ci95_lower',
        'train_degradation_coefficient_ci95_upper',
        'test_time_aupmc',
        'test_time_aupmc_ci95_lower',
        'test_time_aupmc_ci95_upper',
        'test_degradation_coefficient',
        'test_degradation_coefficient_ci95_lower',
        'test_degradation_coefficient_ci95_upper',
        'best_fixed_train_aupmc',
        'best_fixed_train_aupmc_ci95_lower',
        'best_fixed_train_aupmc_ci95_upper',
        'best_fixed_train_missing_prop',
        'minimum_degradation_coefficient',
        'minimum_degradation_coefficient_ci95_lower',
        'minimum_degradation_coefficient_ci95_upper',
    ]]
    .rename(columns={
        'model_name': 'Model',
        'is_distillation_method': 'Distillation method',
        'baseline_auc': 'Baseline AUC',
        'train_time_aupmc': 'Train-time AUPMC',
        'train_time_aupmc_ci95_lower': 'Train-time AUPMC CI95 lower',
        'train_time_aupmc_ci95_upper': 'Train-time AUPMC CI95 upper',
        'train_degradation_coefficient': 'Train degradation coefficient',
        'train_degradation_coefficient_ci95_lower': 'Train degradation coefficient CI95 lower',
        'train_degradation_coefficient_ci95_upper': 'Train degradation coefficient CI95 upper',
        'test_time_aupmc': 'Test-time AUPMC',
        'test_time_aupmc_ci95_lower': 'Test-time AUPMC CI95 lower',
        'test_time_aupmc_ci95_upper': 'Test-time AUPMC CI95 upper',
        'test_degradation_coefficient': 'Test degradation coefficient',
        'test_degradation_coefficient_ci95_lower': 'Test degradation coefficient CI95 lower',
        'test_degradation_coefficient_ci95_upper': 'Test degradation coefficient CI95 upper',
        'best_fixed_train_aupmc': 'Best fixed-train AUPMC',
        'best_fixed_train_aupmc_ci95_lower': 'Best fixed-train AUPMC CI95 lower',
        'best_fixed_train_aupmc_ci95_upper': 'Best fixed-train AUPMC CI95 upper',
        'best_fixed_train_missing_prop': 'Selected train missingness',
        'minimum_degradation_coefficient': 'Minimum degradation coefficient',
        'minimum_degradation_coefficient_ci95_lower': 'Minimum degradation coefficient CI95 lower',
        'minimum_degradation_coefficient_ci95_upper': 'Minimum degradation coefficient CI95 upper',
    })
)

configured_distillation_models = [
    model for model in DISTILLATION_MODEL_NAMES
    if model in set(method_level_metrics_df['model_name'].astype(str))
]
if configured_distillation_models:
    print(
        'Distillation methods excluded from Train-time AUPMC and Train degradation coefficient: '
        + ', '.join(configured_distillation_models)
    )
    print('For distillation methods, train missingness in both/best fixed-train settings refers to student missingness.')

print('Method-level metrics:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000):
    display(method_level_display_df.round(4))

print('Methods ordered by metric:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000):
    display(method_metric_ordering_df)


## Condition-level metrics

Pairwise significant `ΔAUC` matrices by missingness condition.
Within each panel, methods are ordered by condition-specific mean AUC, so rank positions may change from one condition to another.
Only pairs that remain significant after Wilcoxon signed-rank testing with Benjamini-Hochberg FDR correction are highlighted.


In [ ]:
level2_pairwise_df = compute_level2_pairwise_tests(
    level1_df=level1_summary_df,
    replicate_auc_df=replicate_auc_df,
)
level2_plot_df = select_level2_plot_pairs(level2_pairwise_df)

root_tag = RESULTS_ROOT.parent.name
location_tag = TRAIN_DEGRADING_MODALITY.lower()
model_count_tag = f"{level1_summary_df['model_name'].nunique()}models" if not level1_summary_df.empty else '0models'

level2_significant_pairs_df = (
    level2_pairwise_df.loc[level2_pairwise_df['significant_fdr_0p05']]
    .sort_values(['winner_model', 'loser_model', 'train_missing_prop', 'test_missing_prop'])
    .reset_index(drop=True)
)

top_equivalent_group_counts_df = build_top_equivalent_group_counts(level2_plot_df)
general_results_summary_df = build_general_results_summary(
    method_level_metrics_df=method_level_metrics_df,
    level2_plot_df=level2_plot_df,
)

level2_significant_path = OUTPUT_DIR / 'wilcoxon_significant.csv'
top_equivalent_counts_path = OUTPUT_DIR / 'top_equivalent_group_counts.csv'
general_results_summary_path = OUTPUT_DIR / 'general_results_summary.csv'
level2_significant_pairs_df.to_csv(level2_significant_path, index=False)
top_equivalent_group_counts_df.to_csv(top_equivalent_counts_path, index=False)
general_results_summary_df.to_csv(general_results_summary_path, index=False)
print('Saved:', level2_significant_path)
print('Saved:', top_equivalent_counts_path)
print('Saved:', general_results_summary_path)
print('Significant pairwise comparisons:', len(level2_significant_pairs_df))
print('Condition-summary heatmap cells:', len(level2_plot_df))

if HAVE_MPL:
    plot_level3_pairwise_condition_matrices(
        level2_pairwise_df=level2_pairwise_df,
        title=f'Condition-level Pairwise Significant ΔAUC Matrices | ordered by within-condition mean AUC | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME}',
        figures_dir=FIGURES_DIR,
        file_name=f'level2_pairwise_condition_matrices_{root_tag}_{location_tag}_{model_count_tag}.png',
    )
else:
    print('Matplotlib not available; skipping condition-level pairwise ΔAUC matrices.')



### Top equivalent group vs first significantly lower-ranked method

Condition-level summary heatmap derived from the same Wilcoxon signed-rank + FDR results.
Each cell shows the top equivalent group of methods that are all significantly better than the first lower-ranked loser; the number in parentheses gives the loser position in the within-condition mean-AUC ranking.


In [ ]:
if HAVE_MPL:
    plot_level2_significant_pairs_heatmap(
        level2_plot_df=level2_plot_df,
        title=f'Condition-level Top Equivalent Group vs First Significantly Lower-Ranked Method | Wilcoxon Signed-Rank + FDR | {DISPLAY_DATASET_NAME} | endpoint = {ENDPOINT_NAME}',
        figures_dir=FIGURES_DIR,
        file_name=f'level3_significant_pairs_{root_tag}_{location_tag}_{model_count_tag}.png',
    )
else:
    print('Matplotlib not available; skipping condition-level summary heatmap.')


## General results summary

Compact scenario-level summary derived from the method-level metrics and condition-level top-equivalent groups.


In [ ]:
print('General results summary:')
display(general_results_summary_df)

print('Top-equivalent group counts:')
display(top_equivalent_group_counts_df)
